# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print the main metadata (name, description)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All entities are referenced by their `@id`.

Let's list all record sets, their `@id`, and for each their fields and columns:

In [ ]:
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record sets:\n")
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    if hasattr(rs, 'fields') and rs.fields:
        print(f"  Fields:")
        for fld in rs.fields:
            print(f"    - {fld.name} (@id: {fld.id}, dataType: {fld.data_type if hasattr(fld,'data_type') else None})")
    if hasattr(rs, 'columns') and rs.columns:
        print(f"  Columns:")
        for col in rs.columns:
            print(f"    - {col.name} (@id: {col.id}, dataType: {col.data_type if hasattr(col,'data_type') else None})")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Gather all record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    print(f"Loading records for RecordSet {rs_id} ...")
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"{rs_id}: Shape {dataframes[rs_id].shape}")
        print(f"Columns: {dataframes[rs_id].columns.tolist()[:10]}")
        print(dataframes[rs_id].head(2))
    else:
        print(f"No records extracted for {rs_id}.")

# For reproducibility below, pick the first record set with data
main_rs_id = None
for rs_id in record_set_ids:
    if rs_id in dataframes and not dataframes[rs_id].empty:
        main_rs_id = rs_id
        break

print(f"Chosen main record set for further EDA: {main_rs_id}")

# Show full column list for the chosen record set
if main_rs_id:
    print("Columns in df:", dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations include removing outliers, transforming data distributions, or grouping data by key attributes.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Choose a numeric field from the columns (e.g., 'MW' for molecular weight)
# List all numeric-like fields from columns
if main_rs_id:
    numeric_fields = [col for col in dataframes[main_rs_id].columns if dataframes[main_rs_id][col].dtype in [np.float64, np.float32, np.int64, np.int32] or np.issubdtype(dataframes[main_rs_id][col].dtype, np.number)]
    if not numeric_fields:
        # Try to detect numeric columns (sometimes loaded as strings)
        for col in dataframes[main_rs_id].columns:
            # count numbers in first 20 rows
            try:
                sample = dataframes[main_rs_id][col].dropna().astype(float)
                if len(sample) and not np.all(np.isnan(sample)):
                    numeric_fields.append(col)
            except:
                continue

    print(f"Numeric fields detected: {numeric_fields}")

    if numeric_fields:
        numeric_field = numeric_fields[0]  # For illustration, choose the first numeric column
        print(f"Using {numeric_field} for filtering and normalization.")
        # Ensure the dtype is numeric for proper processing
        dataframes[main_rs_id][numeric_field] = pd.to_numeric(dataframes[main_rs_id][numeric_field], errors='coerce')
        threshold = dataframes[main_rs_id][numeric_field].mean()
        filtered_df = dataframes[main_rs_id][dataframes[main_rs_id][numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}, shape: {filtered_df.shape}")
        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to group by a categorical field, e.g., one named 'Sample' or first string field
        group_fields = [col for col in dataframes[main_rs_id].columns if col.lower().startswith('sample') or dataframes[main_rs_id][col].dtype == object]
        group_field = group_fields[0] if group_fields else None
        print(f"Group field chosen: {group_field}")
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped mean {numeric_field} by {group_field}:")
            print(grouped_df.head())
    else:
        print("No numeric field found for EDA.")
else:
    print('No data found in main record set.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

# Histogram and boxplot for the main numeric field
if main_rs_id and 'numeric_field' in locals():
    if numeric_field in dataframes[main_rs_id].columns:
        plt.figure(figsize=(12, 5))
        plt.subplot(1, 2, 1)
        dataframes[main_rs_id][numeric_field].dropna().hist(bins=40)
        plt.title(f"Histogram: {numeric_field}")
        plt.xlabel(numeric_field)
        plt.ylabel('Frequency')

        plt.subplot(1, 2, 2)
        dataframes[main_rs_id][numeric_field].dropna().plot.box(vert=False)
        plt.title(f"Boxplot: {numeric_field}")
        plt.tight_layout()
        plt.show()
        
    # If there is at least one non-numeric field, try a scatter with the second numeric field if available
    if len(numeric_fields) > 1:
        yfield = numeric_fields[1]
        plt.figure(figsize=(6,4))
        plt.scatter(
            dataframes[main_rs_id][numeric_field],
            dataframes[main_rs_id][yfield], alpha=0.5)
        plt.title(f"{numeric_field} vs {yfield}")
        plt.xlabel(numeric_field)
        plt.ylabel(yfield)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We successfully loaded the dataset metadata and explored all record sets.
- Data extraction from record sets enabled quick review of fields and their types.
- Using the chosen numeric field, we performed filtering and normalization, and conducted grouping operations based on available categorical fields.
- Visualizations such as histograms and boxplots provided insight into the distributions of main numeric variables.

Further domain-specific analyses can be tailored to the experimental workflow or intended downstream applications for this mass spectrometry dataset.